# EXP-028 Stage 1 cache audit and optional missing-state extraction

The default Run all performs the complete **CPU audit** directly on the mounted Drive folder. A CPU runtime is sufficient. The notebook exports exact compatible coverage, invalid entries, missing key hashes, and remaining forward counts. The paths below refer to the existing `exp023_handoff.zip` and EXP-026 cache.

After separate owner authorization, switch to an L4 and set `AUTHORIZE_EXPERIMENT=True` for the optional extraction. The worker audits the complete folder before loading the pinned pretrained Qwen, then computes only missing target-language states under disabled adapters. The new cache persists in an isolated directory on Drive. Resume with the same source and inputs. The original cache stays read-only. Cache integrity failures need investigation before extraction.

The 2.5-hour allocation limit includes setup, the full audit, model loading and export. The notebook exports small diagnostics in `outputs.zip` with checksums. Completed state arrays remain separately on Drive under `delta_<source>`. Copy the audited original and delta folders to the CPU workspace for independent verification. This notebook ends after cache verification and export.


In [ ]:
# OWNER_CONFIGURATION
AUTHORIZE_EXPERIMENT = False
AUTHORIZE_GUARDED_CONFIRMATION = False
EXECUTION_SCOPE = "RESEARCH_ONLY_CV"  # Shared scope enum. This notebook only audits or extracts.
RESOURCE_ELIGIBILITY_STATUS = "RESEARCH_ONLY_COMPETITION_UNCONFIRMED"
REQUIRE_SUBMISSION_ZIP = False
EVAL_INPUT_CSV = None
EVAL_DATA_DIR = None
MAX_GPU_HOURS = 2.5
MAX_TOTAL_WALL_HOURS = 2.5
AUTO_BROWSER_DOWNLOAD = True
AUTO_RELEASE_COLAB = True
INPUT_ZIP = "/content/drive/MyDrive/exp023_handoff.zip"
ORIGINAL_CACHE = "/content/drive/MyDrive/exp026/features/8f1e57257da52d7212b0"
PERSISTENT_ROOT = "/content/drive/MyDrive/exp028_stage1_cache"
LOCAL_ROOT = "/content/exp028_stage1"
SOURCE_REVISION = "dae79de30a58909150a747f5c39c48715629248f"


In [ ]:
import hashlib
import json
import math
import os
from pathlib import Path
import subprocess
import sys
import time
import zipfile


def run_pipeline(settings, drive, files, runtime):
    started = time.monotonic()
    local = Path(settings["LOCAL_ROOT"])
    persistent = Path(settings["PERSISTENT_ROOT"])
    attempt = persistent / "attempts" / str(time.time_ns())
    source = local / ("source_" + settings["SOURCE_REVISION"][:12])
    delta = persistent / ("delta_" + settings["SOURCE_REVISION"][:12])
    authorized = settings["AUTHORIZE_EXPERIMENT"] is True
    result = {"status": "STARTING", "browser_arrival": "UNVERIFIED"}
    mounted = False

    def digest(path):
        with path.open("rb") as handle:
            return hashlib.file_digest(handle, "sha256").hexdigest()

    def budget():
        limit = min(settings["MAX_GPU_HOURS"], settings["MAX_TOTAL_WALL_HOURS"]) if authorized else settings["MAX_TOTAL_WALL_HOURS"]
        seconds = float(limit) * 3600 - (time.monotonic() - started) - 180
        if seconds <= 91:
            raise TimeoutError("Allocation exhausted. Completed cache files persist on Drive.")
        return seconds

    def command(args, name, env=None):
        print(f"Running {name}", flush=True)
        path = attempt / (name + ".log")
        with path.open("w") as log:
            run = subprocess.run(args, cwd=local, env=env, timeout=budget(), stdout=log, stderr=subprocess.STDOUT)
        if run.returncode:
            raise RuntimeError(f"{name} stopped with exit code {run.returncode}. Inspect {path}.")

    try:
        drive.mount("/content/drive")
        mounted = True
        attempt.mkdir(parents=True)
        if (settings["EXECUTION_SCOPE"] != "RESEARCH_ONLY_CV"
                or settings["RESOURCE_ELIGIBILITY_STATUS"] != "RESEARCH_ONLY_COMPETITION_UNCONFIRMED"
                or settings["AUTHORIZE_GUARDED_CONFIRMATION"]
                or settings["REQUIRE_SUBMISSION_ZIP"]
                or settings["EVAL_INPUT_CSV"] is not None or settings["EVAL_DATA_DIR"] is not None):
            raise ValueError("Only the frozen research cache audit and extraction scope is supported.")
        for value in (settings["MAX_GPU_HOURS"], settings["MAX_TOTAL_WALL_HOURS"]):
            if not math.isfinite(value) or not 0 < value <= 2.5:
                raise ValueError("Allocation limits must be finite, positive and at most 2.5 hours.")
        local.mkdir(parents=True, exist_ok=True)
        if not Path(settings["ORIGINAL_CACHE"]).is_dir():
            raise FileNotFoundError("Original Drive cache directory is missing. Check ORIGINAL_CACHE.")
        if not Path(settings["INPUT_ZIP"]).is_file():
            raise FileNotFoundError("Upload or locate the unchanged exp023_handoff.zip in Drive.")
        if not source.exists():
            repo = "https://github.com/bqa100507-spec/SemEval-2027-Task-12---ArtEmotion-28-Subtask-3---Image-Only-Cross-Cultural-Sentiment-Prediction.git"
            command(["git", "clone", "--no-checkout", repo, str(source)], "clone")
            command(["git", "-C", str(source), "checkout", "--detach", settings["SOURCE_REVISION"]], "checkout")
        budget()
        actual = subprocess.check_output(["git", "-C", str(source), "rev-parse", "HEAD"], timeout=15, text=True).strip()
        if actual != settings["SOURCE_REVISION"]:
            raise ValueError("The existing Colab checkout has a different source revision.")
        env = dict(os.environ, UV_PYTHON_INSTALL_DIR=str(local / "python"), UV_CACHE_DIR=str(local / "uv_cache"),
                   HF_HUB_DISABLE_XET="1")
        command([sys.executable, "-m", "pip", "install", "uv==0.7.13"], "uv", env)
        uv = [sys.executable, "-m", "uv"]
        command([*uv, "python", "install", "3.12.10"], "python", env)
        venv = local / ("extract_env" if authorized else "audit_env")
        python = str(venv / "bin/python")
        if not venv.exists():
            command([*uv, "venv", "--python", "3.12.10", str(venv)], "venv", env)
        if authorized:
            command([*uv, "pip", "sync", "--python", python, "--extra-index-url",
                     "https://download.pytorch.org/whl/cu126", "--index-strategy", "unsafe-best-match",
                     str(source / "requirements-exp026.lock.txt")], "dependencies", env)
        else:
            command([*uv, "pip", "install", "--python", python, "numpy==2.2.6", "pandas==2.2.3",
                     "Pillow==11.2.1", "scipy==1.15.3", "scikit-learn==1.6.1", "threadpoolctl==3.6.0",
                     "pytest==8.3.5", "psutil==7.2.2"], "dependencies", env)
        command([*uv, "pip", "check", "--python", python], "dependency_check", env)
        cpu_env = dict(env, CUDA_VISIBLE_DEVICES="-1")
        tests = [str(source / "tests/test_exp028_stage1.py"), str(source / "tests/test_experiment_workflow.py")]
        # pytest's root discovery starts at the tests in the pinned checkout.
        command([python, "-m", "pytest", "-q", *tests], "CPU_workflow_checks", cpu_env)
        worker = [python, "-u", str(source / "scripts/extract_exp028_missing.py"),
                  "--stage", "extract" if authorized else "audit",
                  "--input-zip", settings["INPUT_ZIP"], "--inputs", str(local / "inputs"),
                  "--qwen-cache", settings["ORIGINAL_CACHE"], "--delta-root", str(delta),
                  "--model-cache", str(local / "model"), "--output", str(attempt / "verification"),
                  "--revision", actual, "--max-seconds", str(min(9000, budget()-5))]
        command(worker + (["--authorize-extraction"] if authorized else []), "cache_work", env if authorized else cpu_env)
        result.update(json.loads((attempt / "verification/result.json").read_text()))
        audits = [p for p in (attempt / "verification").glob("*audit.json")]
        for path in sorted(audits):
            audit = json.loads(path.read_text())
            keys = ("verified_vision_images", "verified_language_states", "missing_vision_images",
                    "missing_language_states", "invalid_entries", "remaining_forwards_relative_to_supplied_cache")
            print(path.name, json.dumps({k: audit[k] for k in keys}, indent=2))
    except Exception as error:
        result.update(status="STOPPED", error=f"{type(error).__name__}: {error}")
    finally:
        try:
            if mounted:
                attempt.mkdir(parents=True, exist_ok=True)
                (attempt / "status.json").write_text(json.dumps(result, indent=2))
                payload = {p.relative_to(attempt).as_posix(): digest(p) for p in sorted(attempt.rglob("*")) if p.is_file()}
                output = attempt / "outputs.zip"
                with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as archive:
                    for name in payload:
                        archive.write(attempt / name, name)
                    archive.writestr("manifest_sha256.json", json.dumps(payload, indent=2))
                with zipfile.ZipFile(output) as archive:
                    if any(hashlib.sha256(archive.read(n)).hexdigest() != h for n, h in payload.items()):
                        raise IOError("Persistent output ZIP read-back failed.")
                result["persistent_readback"] = "PASS"
                result["receipt_zip_sha256"] = digest(output)
                if settings["AUTO_BROWSER_DOWNLOAD"]:
                    result["download_request"] = "REQUESTED"
                    (attempt / "download_attempt.json").write_text(json.dumps(result, indent=2))
                    try:
                        files.download(str(output))
                        result["download_request"] = "API_RETURNED"
                    except Exception as error:
                        result["download_request"] = f"FAILED: {error}"
        except Exception as error:
            result.update(export_error=str(error), persistent_readback="FAIL")
        finally:
            result["runtime_release"] = "REQUESTED_NOT_CONFIRMED" if settings["AUTO_RELEASE_COLAB"] else "DISABLED"
            try:
                if mounted:
                    (attempt / "final_report.json").write_text(json.dumps(result, indent=2))
                print(json.dumps(result, indent=2))
            finally:
                if settings["AUTO_RELEASE_COLAB"]:
                    runtime.unassign()
    return result


In [ ]:
from google.colab import drive, files, runtime

run_cloud = run_pipeline
result = run_cloud(globals(), drive, files, runtime)
